# 06 Evaluate Flood Outputs

## Stage Contract

Requires:
- completed SFINCS scenario outputs
- Wflow readiness report
- SMART-DS evaluation footprint and asset source

Produces:
- completed-event stats table
- multi-domain depth merge diagnostics
- SMART-DS asset evaluation outputs

## Imports and Runtime


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml

location_root = Path("..").resolve()
location_name = location_root.name
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

from wflow_runs.notebook import load_runtime
runtime = load_runtime(location_root, wflow_domain_review_required=True)
location_root = runtime.location_root
location_name = runtime.location_name
runtime_config = runtime.runtime_config
config = runtime.config
grid_config = runtime.config
data_sources = runtime.config
sfincs_config = runtime.sfincs_config
wflow_config = runtime.wflow_config
resolve_location_path = runtime.resolve_location_path
ensure_parent = runtime.ensure_parent

pd.set_option("display.max_colwidth", 120)


## Stats Inputs


In [ ]:
from wflow_runs.notebook import exists_table
exists_table(location_root, {
    "stats root": sfincs_config["paths"]["stats_root"],
    "run outputs root": sfincs_config["paths"]["storage_root"],
    "evaluation footprint": grid_config["smart_ds_evaluation_footprint"]["output"],
    "asset source": sfincs_config["evaluation"]["asset_source"],
})


## Wflow Readiness


In [ ]:
readiness_report = wflow_config["wflow"]["readiness_validation"]["report"]
status = exists_table(location_root, {"wflow readiness report": readiness_report})
status["production_gate"] = wflow_config["wflow"]["readiness_validation"]["decision"]
status


## Completed Events


In [ ]:
scenario_catalog_path = resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv")
scenario_build_report_path = resolve_location_path(sfincs_config["paths"]["scenarios_root"]) / "scenario_build_report.csv"
scenario_catalog = pd.read_csv(scenario_catalog_path) if scenario_catalog_path.exists() else pd.DataFrame()
scenario_build_report = pd.read_csv(scenario_build_report_path) if scenario_build_report_path.exists() else pd.DataFrame()

if not scenario_catalog.empty and not scenario_build_report.empty:
    staged = (
        scenario_build_report.groupby("event_id", as_index=False)
        .agg(domain_count=("run_root", "count"), first_run_root=("run_root", "first"), scenario_status=("scenario_status", "first"))
    )
    completed_events = scenario_catalog.merge(staged, on="event_id", how="left")
else:
    completed_events = scenario_catalog.copy()

if completed_events.empty:
    completed_events = pd.DataFrame(columns=["event_id", "event_origin", "catalog_role", "severity_band", "scenario_status"])
for col in ["event_origin", "catalog_role", "severity_band"]:
    if col not in completed_events:
        completed_events[col] = "unresolved"
    completed_events[col] = completed_events[col].fillna("unresolved").astype(str)

display(completed_events.groupby(["event_origin", "catalog_role", "severity_band"], dropna=False).size().rename("event_count").reset_index())
completed_events.head(12)


## Multi-Domain Depth Merge


In [ ]:
merge_settings = sfincs_config["evaluation"]["multi_domain_merge"]
pd.Series({
    "method": merge_settings["method"],
    "retain_source_domain_id": merge_settings["retain_source_domain_id"],
    "write_overlap_diagnostics": merge_settings["write_overlap_diagnostics"],
    "sfincs_domain_merge": sfincs_config["sfincs_domain_set"]["evaluation_merge"],
})


## SMART-DS Asset Evaluation


In [ ]:
pd.Series({
    "asset_source": sfincs_config["evaluation"]["asset_source"],
    "output_root": sfincs_config["evaluation"]["output_root"],
    "minimum_flood_coverage": grid_config["smart_ds_evaluation_footprint"]["minimum_flood_coverage"],
    "evaluation_footprint": grid_config["smart_ds_evaluation_footprint"]["output"],
})


## Alignment QA


In [ ]:
merge_settings = sfincs_config["evaluation"]["multi_domain_merge"]
pd.DataFrame([
    {"check": "shared event catalog row used across domains", "expected": True, "observed": sfincs_config["sfincs_domain_set"]["event_catalog_scope"] == "shared_across_domain_set"},
    {"check": "Wflow event scope shared across submodels", "expected": True, "observed": wflow_config["wflow"]["domain_set"]["event_catalog_scope"] == "shared_across_domain_set"},
    {"check": "source domain retained in merged depths", "expected": True, "observed": merge_settings["retain_source_domain_id"]},
])


## Write Notebook Outputs


In [ ]:
write_outputs = False
evaluation_root = resolve_location_path(sfincs_config["evaluation"]["output_root"])
catalogue_path = evaluation_root / "flood_event_outcome_catalogue.csv"
if write_outputs:
    evaluation_root.mkdir(parents=True, exist_ok=True)
    completed_events.to_csv(catalogue_path, index=False)
pd.Series({
    "write_outputs": write_outputs,
    "catalogue_csv": str(catalogue_path),
    "catalogue_rows": len(completed_events) if "completed_events" in globals() else 0,
    "target_root": str(evaluation_root),
})
